In [ ]:
import h5py
file_path = '/share/rcif2/mmangat/logs/HF-900MeV-eta4_20260305-T105911/ckpts/epoch=022-val_loss=0.03377_test_eval.h5'
def print_tree(name, obj):
    indent = "  " * name.count("/")
    if isinstance(obj, h5py.Group):
        print(f"{indent}[{name}]")
    elif isinstance(obj, h5py.Dataset):
        print(f"{indent}{name}  shape={obj.shape} dtype={obj.dtype}")

with h5py.File(file_path, "r") as f:
    print("/")
    f['29800'].visititems(print_tree)

## Load all events

For each hit, collect its `r`, ground truth `is_first`, predicted `is_first`, and the `is_first_prob` score.
We also join the particle `pt` onto each first hit so we can study efficiency vs pT.

In [ ]:
import numpy as np
import pandas as pd

records = []

with h5py.File(file_path, "r") as f:
    for sample_id in f.keys():
        g = f[sample_id]

        r                = g["inputs/hit_r"][0].astype(float)           # (N_hits,)
        eta              = g["inputs/hit_eta"][0].astype(float)
        true_first       = g["targets/hit_is_first"][0]                 # bool (N_hits,)
        true_on_particle = g["targets/hit_on_valid_particle"][0]        # bool (N_hits,)
        pred_first       = g["preds/final/hit_filter/hit_is_first"][0]  # bool (N_hits,)
        prob_first       = g["preds/final/hit_filter/hit_is_first_prob"][0].astype(float)
        hit_particle_id  = g["targets/hit_particle_id"][0]              # int (N_hits,)

        particle_id      = g["targets/particle_id"][0]                  # int (N_particles,)
        particle_valid   = g["targets/particle_valid"][0]               # bool (N_particles,)
        particle_pt      = g["targets/particle_pt"][0]                  # float (N_particles,)

        # Build a map from particle_id -> pt for valid particles only
        pt_map = {pid: pt for pid, pt, valid in zip(particle_id, particle_pt, particle_valid) if valid}

        # One row per hit
        df = pd.DataFrame({
            "r":              r,
            "eta":            eta,
            "true_first":     true_first,
            "true_on_particle": true_on_particle,
            "pred_first":     pred_first,
            "prob_first":     prob_first,
            "particle_id":    hit_particle_id,
            "sample_id":      int(sample_id),
        })
        df["particle_pt"] = df["particle_id"].map(pt_map)
        records.append(df)

hits = pd.concat(records, ignore_index=True)
print(f"Loaded {len(hits):,} hits across {len(records)} events")
print(f"True first hits: {hits['true_first'].sum():,}")
print(f"Pred first hits: {hits['pred_first'].sum():,}")

## Overall classification metrics

Global efficiency (recall) and purity (precision) for the `is_first` classification.

In [ ]:
true = hits["true_first"]
pred = hits["pred_first"]

tp = (true & pred).sum()
fp = (~true & pred).sum()
fn = (true & ~pred).sum()

efficiency = tp / true.sum()   # recall
purity     = tp / pred.sum()   # precision

print(f"True first hits   : {true.sum():,}")
print(f"Predicted first   : {pred.sum():,}")
print(f"TP={tp:,}  FP={fp:,}  FN={fn:,}")
print(f"Efficiency (recall)  : {efficiency:.4f}")
print(f"Purity (precision)   : {purity:.4f}")

## Efficiency vs r of the first hit

Displaced tracks have their first hit at larger `r`. A drop in efficiency at large `r` means we are missing those tracks.

In [ ]:
import matplotlib.pyplot as plt

# Only consider hits that are true first hits on a valid particle
first_hits = hits[hits["true_first"]].copy()

# r is stored in metres (scaled by 0.01 in data.py)
# Convert to mm for readability
first_hits["r_mm"] = first_hits["r"] * 1000

r_bins = np.linspace(first_hits["r_mm"].min(), first_hits["r_mm"].max(), 30)
bin_centres = 0.5 * (r_bins[:-1] + r_bins[1:])

total, _ = np.histogram(first_hits["r_mm"], bins=r_bins)
correct, _ = np.histogram(first_hits.loc[first_hits["pred_first"], "r_mm"], bins=r_bins)

eff = np.where(total > 0, correct / total, np.nan)
eff_err = np.where(total > 0, np.sqrt(eff * (1 - eff) / total), np.nan)

fig, axes = plt.subplots(2, 1, figsize=(8, 7), sharex=True)

axes[0].bar(bin_centres, total, width=np.diff(r_bins), color="steelblue", alpha=0.7, label="True first hits")
axes[0].set_ylabel("Count")
axes[0].set_title("Distribution of true first hits vs r")
axes[0].legend()

axes[1].errorbar(bin_centres, eff, yerr=eff_err, fmt="o-", color="firebrick", capsize=3)
axes[1].axhline(1.0, color="grey", linestyle="--", linewidth=0.8)
axes[1].set_ylim(0, 1.05)
axes[1].set_xlabel("r of true first hit (mm)")
axes[1].set_ylabel("Selection efficiency")
axes[1].set_title("First-hit selection efficiency vs r")

plt.tight_layout()
plt.savefig("efficiency_vs_r.pdf", bbox_inches="tight")
plt.show()

## Efficiency vs |eta| of the first hit

Check for any barrel/endcap asymmetry.

In [ ]:
first_hits["abs_eta"] = first_hits["eta"].abs()

eta_bins = np.linspace(0, first_hits["abs_eta"].max(), 20)
eta_centres = 0.5 * (eta_bins[:-1] + eta_bins[1:])

total_eta, _   = np.histogram(first_hits["abs_eta"], bins=eta_bins)
correct_eta, _ = np.histogram(first_hits.loc[first_hits["pred_first"], "abs_eta"], bins=eta_bins)

eff_eta     = np.where(total_eta > 0, correct_eta / total_eta, np.nan)
eff_eta_err = np.where(total_eta > 0, np.sqrt(eff_eta * (1 - eff_eta) / total_eta), np.nan)

fig, axes = plt.subplots(2, 1, figsize=(8, 7), sharex=True)

axes[0].bar(eta_centres, total_eta, width=np.diff(eta_bins), color="steelblue", alpha=0.7)
axes[0].set_ylabel("Count")
axes[0].set_title("Distribution of true first hits vs |eta|")

axes[1].errorbar(eta_centres, eff_eta, yerr=eff_eta_err, fmt="o-", color="firebrick", capsize=3)
axes[1].axhline(1.0, color="grey", linestyle="--", linewidth=0.8)
axes[1].set_ylim(0, 1.05)
axes[1].set_xlabel("|eta| of true first hit")
axes[1].set_ylabel("Selection efficiency")
axes[1].set_title("First-hit selection efficiency vs |eta|")

plt.tight_layout()
plt.savefig("efficiency_vs_eta.pdf", bbox_inches="tight")
plt.show()

## Efficiency vs particle pT

Check whether low-pT particles (which tend to curl and have shorter tracks) are harder to identify.

In [ ]:
first_hits_with_pt = first_hits.dropna(subset=["particle_pt"])

pt_bins = np.linspace(0, min(first_hits_with_pt["particle_pt"].quantile(0.98), 10), 25)
pt_centres = 0.5 * (pt_bins[:-1] + pt_bins[1:])

total_pt, _   = np.histogram(first_hits_with_pt["particle_pt"], bins=pt_bins)
correct_pt, _ = np.histogram(first_hits_with_pt.loc[first_hits_with_pt["pred_first"], "particle_pt"], bins=pt_bins)

eff_pt     = np.where(total_pt > 0, correct_pt / total_pt, np.nan)
eff_pt_err = np.where(total_pt > 0, np.sqrt(eff_pt * (1 - eff_pt) / total_pt), np.nan)

fig, axes = plt.subplots(2, 1, figsize=(8, 7), sharex=True)

axes[0].bar(pt_centres, total_pt, width=np.diff(pt_bins), color="steelblue", alpha=0.7)
axes[0].set_ylabel("Count")
axes[0].set_title("Distribution of true first hits vs particle pT")

axes[1].errorbar(pt_centres, eff_pt, yerr=eff_pt_err, fmt="o-", color="firebrick", capsize=3)
axes[1].axhline(1.0, color="grey", linestyle="--", linewidth=0.8)
axes[1].set_ylim(0, 1.05)
axes[1].set_xlabel("Particle pT (GeV)")
axes[1].set_ylabel("Selection efficiency")
axes[1].set_title("First-hit selection efficiency vs particle pT")

plt.tight_layout()
plt.savefig("efficiency_vs_pt.pdf", bbox_inches="tight")
plt.show()

## Score distributions

Predicted `is_first` probability for true first hits vs all other hits. A well-separated model will show two distinct peaks.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))

ax.hist(hits.loc[hits["true_first"], "prob_first"],
        bins=50, density=True, alpha=0.6, label="True first hits", color="steelblue")
ax.hist(hits.loc[~hits["true_first"], "prob_first"],
        bins=50, density=True, alpha=0.6, label="Other hits", color="firebrick")

ax.axvline(0.1, color="black", linestyle="--", linewidth=1, label="Threshold (0.1)")
ax.set_xlabel("Predicted is_first probability")
ax.set_ylabel("Normalised density")
ax.set_title("Score distribution: first hits vs other hits")
ax.legend()

plt.tight_layout()
plt.savefig("score_distributions.pdf", bbox_inches="tight")
plt.show()

## Efficiency vs r broken down by detector region

Join with the original parquet files to get `volume_id` and `layer_id`, allowing a clean per-layer breakdown.
Hits are in the same order as the parquet file after the `volume_id` filter, so a positional join works.

In [ ]:
from pathlib import Path

prepped_dir   = Path("/share/rcif2/pduckett/data/prepped/test/")
hit_volume_ids = [7, 8, 9]

layer_records = []

with h5py.File(file_path, "r") as f:
    for sample_id in f.keys():
        event_name = f"event{int(sample_id):09d}"
        parquet_path = prepped_dir / f"{event_name}-hits.parquet"
        if not parquet_path.exists():
            continue

        parquet_hits = pd.read_parquet(parquet_path, columns=["volume_id", "layer_id"])
        parquet_hits = parquet_hits[parquet_hits["volume_id"].isin(hit_volume_ids)].reset_index(drop=True)

        g = f[sample_id]
        true_first = g["targets/hit_is_first"][0]
        pred_first = g["preds/final/hit_filter/hit_is_first"][0]

        parquet_hits["true_first"] = true_first
        parquet_hits["pred_first"] = pred_first
        layer_records.append(parquet_hits)

layer_df = pd.concat(layer_records, ignore_index=True)

# Efficiency per (volume_id, layer_id)
layer_eff = (
    layer_df[layer_df["true_first"]]
    .groupby(["volume_id", "layer_id"])
    .apply(lambda g: pd.Series({
        "efficiency": g["pred_first"].sum() / len(g),
        "n_first_hits": len(g),
    }), include_groups=False)
    .reset_index()
    .sort_values(["volume_id", "layer_id"])
)

print(layer_eff.to_string(index=False))

In [ ]:
volume_labels = {7: "Pixel endcap -z (vol 7)", 8: "Pixel barrel (vol 8)", 9: "Pixel endcap +z (vol 9)"}
colors        = {7: "darkorange", 8: "steelblue", 9: "seagreen"}

fig, ax = plt.subplots(figsize=(10, 5))

for vol_id, group in layer_eff.groupby("volume_id"):
    ax.errorbar(
        group["layer_id"],
        group["efficiency"],
        fmt="o-",
        label=volume_labels.get(vol_id, f"vol {vol_id}"),
        color=colors.get(vol_id, "grey"),
        capsize=3,
    )

ax.axhline(1.0, color="grey", linestyle="--", linewidth=0.8)
ax.set_ylim(0, 1.05)
ax.set_xlabel("Layer ID")
ax.set_ylabel("Selection efficiency")
ax.set_title("First-hit selection efficiency per detector layer")
ax.legend()

plt.tight_layout()
plt.savefig("efficiency_per_layer.pdf", bbox_inches="tight")
plt.show()